<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 6 — Dynamics and Geometry
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18439445.svg)](https://doi.org/10.5281/zenodo.18439445)

**Author:** E.S. Brooke · **Paper:** v2.1 main + v1.2 supplement · **Codebase:** v6.9

---

### What this notebook is

Paper 6 is the **spacetime, gravity, and cosmology** paper. **Primary bridge-theorem receiver.** Supplies $d_{\rm eff} = 102$, T12 partition, $C_{\rm vacuum} = 42$ (independent route) to Paper 8.

### What this notebook is **not**

A resolution of the H_0 tension (it is stated as a falsifier, not resolved). A dark-matter particle identification (open). A quantum gravity theory (structural content only).

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity.git 2>/dev/null || true
%cd -q APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 21 bank-registered checks bundled.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · PLEC dynamics — least-cost path form

$$\boxed{\quad S[\gamma] \;=\; \int_\gamma \kappa\,d\tau,\qquad \text{admissible trajectories minimise}\;S. \quad}$$

Euler-Lagrange equations follow; five-type regime-exit taxonomy classifies how trajectories leave $R$.

In [ ]:
show('check_T_PLEC_path_form')

## §3 · Lorentzian signature + 4D spacetime

- **$T_{7B}$** — Lorentzian signature forced.
- **$T_8$** — spacetime dimension $d = 4$ via full six-Δ chain.

In [ ]:
show('check_T_7B')

In [ ]:
show('check_T_8')

## §4 · Einstein equations via $A9$ + Lovelock

**$A9$ unified closure** + **Lovelock's theorem** ⇒ Einstein-Hilbert action is the unique admissible gravitational action (in $d = 4$).

**$T_{\rm graviton}$** via Weinberg-Witten: massless spin-2 graviton, structurally required.

In [ ]:
show('check_A9_closure')

## §5 · $T_{11}$ — the independent route to $C_{\rm vacuum} = 42$

$$\boxed{\quad C_{\rm vacuum} \;=\; \underbrace{27}_{\text{gauge-index}} \;+\; \underbrace{3}_{\text{Higgs internal}} \;+\; \underbrace{12}_{\text{generators}} \;=\; 42 \quad}$$

This is an **independent** derivation of 42. It does not go through the matter-sector subtraction $61 - 3 - 16 = 42$ (that's Paper 3 L_equip). Two independent routes converging on 42 is a real structural fact — see `DO_NOT_CLAIM.md` row 4.

Consequence: $\Omega_\Lambda = C_{\rm vacuum} / C_{\rm total} = 42/61 \approx 0.6885$. Planck 2018: $0.6847 \pm 0.0073$ (0.5σ).

In [ ]:
show('check_T11')

## §6 · Primary bridge receiver (Supp §Bridge_compressed)

$$\boxed{\begin{aligned}
&T_{\rm interface\_sector\_bridge}:\; V_{61} = V_{\rm local} \oplus V_{\rm global}\\
&\text{and}\;\; V_{\rm global} = \text{Sector B}_{T_{\rm horizon\_reciprocity}} = V_\Lambda\\
&\text{so}\;\; d_{\rm eff} = |V_{61} \setminus \text{self}| + \dim V_{\rm global} = 60 + 42 = 102
\end{aligned}}$$

The two "42"s in $T_{11}$ (cosmological) and $L_{\rm self\_exclusion}$ (horizon) are the **same 42-dim subspace** $V_{\rm global}$.

In [ ]:
show('check_T_interface_sector_bridge')

In [ ]:
show('check_L_self_exclusion')

## §7 · $H_0$ prediction and the 7.09σ falsifier

APF predicts $H_0 = 67.76$ km/s/Mpc (consistent with Planck 2018).

H0DN 2026: $H_0 = 73.50 \pm 0.81$. **Tension: 7.09σ.**

This is stated as a **framework falsifier** in Main §11.4, not a resolution. Routes I-IV excluded; Route V (local inhomogeneity) is the only APF-admissible path but insufficient by ~2×.

See `DO_NOT_CLAIM.md` rows 1, 2 before writing about this.

## §8 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18529115)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper6_2026,
  author  = {Brooke, Ethan S.},
  title   = {Dynamics and Geometry as Optimal Admissible Reallocation},
  year    = 2026,
  version = {v2.1 main + v1.2 supplement},
  doi     = {10.5281/zenodo.18439445}
}
```

*Paper-companion repo · v2.1 main + v1.2 supplement · Phase 22 gorgeous-math edition · 2026-04-24.*